In [1]:
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
from tqdm import tqdm
import heapq
import numpy as np

with open('./full_graph_split1_eval.pkl', 'rb') as f:
    res = pickle.load(f)
result = pd.DataFrame(res['result'])

with open('./graphmask_output_indication.pkl', 'rb') as f:
    d_gm = pickle.load(f)

In [2]:
### filtering CYP genes

def preprocess(d):
    d = d[~d.y_name.str.contains('CYP')]
    d = d[~d.x_name.str.contains('CYP')]
    d = d.rename(columns = {'indication_layer1_att': 'layer1_att', 'indication_layer2_att': 'layer2_att'})
    return d

## filtering not-that-interesting meta paths?
not_cool_rel = ['rev_contraindication', 'contraindication', 'drug_drug', 'rev_off-label use', 'off-label use', 'anatomy_protein_absent', 'rev_anatomy_protein_absent']

d_gm = preprocess(d_gm)

In [3]:
def build_graph(df):
    G = nx.MultiDiGraph()
    for _, row in tqdm(df.iterrows()):
        # Add nodes with their types
        G.add_node(row['x_name'], node_type=row['x_type'])
        G.add_node(row['y_name'], node_type=row['y_type'])

        # Add edges with relation type and weights
        G.add_edge(row['x_name'], row['y_name'], relation=row['relation'],
                   layer1_att=row['layer1_att'], layer2_att=row['layer2_att'])
    return G

def get_two_hop_neighborhood(G, node_id):
    # Get neighbors within two hops
    neighbors_one_hop = set(nx.all_neighbors(G, node_id))
    neighbors_two_hop = set()
    for n in neighbors_one_hop:
        neighbors_two_hop.update(nx.all_neighbors(G, n))
    return neighbors_one_hop.union(neighbors_two_hop)

def find_relation_specific_paths(G, start_id, end_id, max_depth=4):
    paths = []
    for path in nx.all_simple_paths(G, source=start_id, target=end_id, cutoff=max_depth):
        # Construct relation-specific path
        relation_path = [start_id]
        for i in range(len(path) - 1):
            relation = G[path[i]][path[i + 1]][0]['relation']  # Assuming one relation per edge for simplicity
            relation_path.extend([relation, path[i + 1]])
        paths.append(relation_path)
    return paths

def get_two_hop_neighborhood_enrichment_per_relation(G, node_id, K, K2, relation_averages, enrichment = False):
    
    # First hop: Get the top K neighbors for each relation type
    neighbors_by_relation = {}
    for neighbor in G[node_id]:
        for edge_key in G[node_id][neighbor]:
            edge_data = G[node_id][neighbor][edge_key]
            relation = edge_data['relation']
            weight = edge_data['layer1_att'] + edge_data['layer2_att']
            if enrichment:
                avg_weight = relation_averages[relation]
                relative_increase = ((weight - avg_weight) / avg_weight) * 100
            else:
                relative_increase = weight
            if relation not in neighbors_by_relation:
                neighbors_by_relation[relation] = []
            heapq.heappush(neighbors_by_relation[relation], (-relative_increase, neighbor))

    # Select the top K neighbors for each relation type
    first_hop_neighbors = set()
    for relation, neighbors in neighbors_by_relation.items():
        top_neighbors = [neighbor for _, neighbor in heapq.nlargest(K, neighbors)]
        first_hop_neighbors.update(top_neighbors)

    # Second hop: Repeat the process for each neighbor in first_hop_neighbors
    second_hop_neighbors = set()
    for first_hop_neighbor in first_hop_neighbors:
        neighbors_by_relation = {}
        for neighbor in G[first_hop_neighbor]:
            for edge_key in G[first_hop_neighbor][neighbor]:
                edge_data = G[first_hop_neighbor][neighbor][edge_key]
                relation = edge_data['relation']
                weight = edge_data['layer1_att'] + edge_data['layer2_att']
                
                if enrichment:
                    avg_weight = relation_averages[relation]
                    relative_increase = ((weight - avg_weight) / avg_weight) * 100
                else:
                    relative_increase = weight

                if relation not in neighbors_by_relation:
                    neighbors_by_relation[relation] = []
                heapq.heappush(neighbors_by_relation[relation], (-relative_increase, neighbor))

        for relation, neighbors in neighbors_by_relation.items():
            top_neighbors = [neighbor for _, neighbor in heapq.nlargest(K, neighbors)]
            second_hop_neighbors.update(top_neighbors)

    return first_hop_neighbors.union(second_hop_neighbors)

def score_path_enrichment(G, path, relation_averages, enrichment = False):
    score = 0
    path_length = len(path) // 2  # Number of edges in the path

    for i in range(0, path_length * 2, 2):
        node1 = path[i]
        relation = path[i + 1]
        node2 = path[i + 2]

        if (node1, node2) in G.edges():
            for edge_key in G[node1][node2]:
                edge_data = G[node1][node2][edge_key]
                if edge_data['relation'] == relation:
                    weight = edge_data['layer1_att'] + edge_data['layer2_att']
                    if enrichment:
                        avg_weight = relation_averages[relation]
                        # Calculate percentage relative increase
                        relative_increase = ((weight - avg_weight) / avg_weight) * 100
                    else:
                        relative_increase = weight
                    score += relative_increase

    score /= path_length

    return score

def print_beautiful_path(path):
    path = [i if 'rev' not in i else i[4:] for i in path]
    return ' -> '.join(path)

def print_beautiful_paths(paths):
    return [print_beautiful_path(i) for i in paths]

def group_paths_by_meta_paths_with_node_types(paths, G):
    meta_paths_dict = {}

    for path in paths:
        # Extract the meta path with node types and relation types
        meta_path = []
        for i in range(len(path)):
            if i % 2 == 0:  # Node
                node_id = path[i]
                node_type = G.nodes[node_id]['node_type']
                meta_path.append(node_type)
            else:  # Relation
                meta_path.append(path[i])

        meta_path = tuple(meta_path)

        # Add the path to the list of paths for the corresponding meta path
        if meta_path not in meta_paths_dict:
            meta_paths_dict[meta_path] = []
        meta_paths_dict[meta_path].append(path)

    return meta_paths_dict

def calculate_relation_averages(G, layer_only = False):
    relation_sums = {}
    relation_counts = {}

    # Iterate over all edges in the graph
    for u, v, data in tqdm(G.edges(data=True)):
        relation = data['relation']
        if layer_only:
            weight = data['layer' + str(layer_only) + '_att']
        else:
            weight = data['layer1_att'] + data['layer2_att']
        
        # Summing weights and counting occurrences for each relation
        if relation in relation_sums:
            relation_sums[relation] += weight
            relation_counts[relation] += 1
        else:
            relation_sums[relation] = weight
            relation_counts[relation] = 1

    # Calculating averages
    relation_averages = {rel: relation_sums[rel] / relation_counts[rel] for rel in relation_sums}
    return relation_averages

In [4]:
def find_meta_paths(X_id, Y_id, G, not_cool_rel, relation_averages, enrichment = False):
    neighborhood_X = get_two_hop_neighborhood_enrichment_per_relation(G, X_id, K, K2, relation_averages, enrichment)
    neighborhood_Y = get_two_hop_neighborhood_enrichment_per_relation(G, Y_id, K, K2, relation_averages, enrichment)
    common_neighborhood = neighborhood_X.union(neighborhood_Y)
    subG = G.subgraph(common_neighborhood)
    #print('Number of neighbors: ', len(subG))
    paths = find_relation_specific_paths(subG, X_id, Y_id, max_depth=4)
    path_scores_enrichment = [score_path_enrichment(G, path, relation_averages, enrichment) for path in paths]
    meta_paths_dict = group_paths_by_meta_paths_with_node_types(paths, G)
    meta_paths = list(meta_paths_dict.keys())
    valid_meta_paths = [i for i in meta_paths if len(np.intersect1d(not_cool_rel, i)) == 0]
    return paths, path_scores_enrichment, meta_paths_dict, meta_paths, valid_meta_paths

In [5]:
import pickle
with open('name_mapping.pkl', 'rb') as f: 
    mapping = pickle.load(f)

In [6]:
idx2id_disease = mapping['idx2id_disease'] 
idx2id_drug = mapping['idx2id_drug'] 
id2name_disease = mapping['id2name_disease'] 
id2name_drug = mapping['id2name_drug']
id2idx_disease = {j:i for i,j in idx2id_disease.items()}
name2id_disease = {j:i for i,j in id2name_disease.items()}
name2id_drug = {j:i for i,j in id2name_drug.items()}

In [7]:
## for each relation, get the top K nodes.
K = 10 ## top K first hop neighbors
K2 = 10 ## top K2 second hop neighbors

In [8]:
## build a graph using either d_att, d_gm, d_ge
G_gm = build_graph(d_gm)

## if use enrichment mode (default FALSE, to calculate average weight per relation for enrichment calculation
relation_averages_gm = calculate_relation_averages(G_gm)

7676670it [15:41, 8157.77it/s] 
100%|█████████████████████████████████████████████████████████████████████████████████| 7676670/7676670 [00:21<00:00, 357746.34it/s]


In [9]:
G_dict = {
    'gm': G_gm,
}

relation_avg_dict = {
    'gm': relation_averages_gm,
}

In [10]:
# top 20 diseases with many connections to the graph but zero indications
# top 20 diseases with few indications (ranked by accuracy)
# 10 diseases with recently approved therapies (which means lots of interest in these diseases)

def sigmoid(x):
    return 1/(1+np.exp(-x))

In [21]:
def get_path(X_id, Y_id, G, not_cool_rel, enrichment, label):
    path = find_meta_paths(X_id, Y_id, G, not_cool_rel, relation_avg_dict['gm'], enrichment = enrichment)
    #print('Number of paths: ', len(path))
    to_save = []
    for meta_path, paths in path[2].items():
        if meta_path in path[-1]:
            path_scores = [score_path_enrichment(G, path, relation_avg_dict['gm'], enrichment) for path in paths]
            to_save += tuple(zip([X_id]* len(path_scores), 
                                 [Y_id]* len(path_scores), 
                                 [print_beautiful_path(meta_path)] * len(path_scores), 
                                 print_beautiful_paths(paths), 
                                 path_scores, 
                                 [sigmoid(res['prediction'][name2id_disease[X_id]][name2id_drug[Y_id]])] * len(path_scores)))
    out =  pd.DataFrame(to_save).rename(columns = {0: 'Disease', 1: 'Drug', 2: 'Meta-Path', 3: 'Path', 4: 'Path Score', 5: 'Prediction Score'})
    out['Category'] = label
    return out

In [22]:
G = G_dict['gm']
enrichment = False
X_id, Y_id = 'macular degeneration', 'Polidocanol'
label = 'example'
get_path(X_id, Y_id, G, not_cool_rel, enrichment, label)

,Disease,Drug,Meta-Path,Path,Path Score,Prediction Score,Category
0,macular degeneration,Polidocanol,disease -> disease_phenotype_positive -> effec...,macular degeneration -> disease_phenotype_posi...,0.778900,0.993007,example
1,macular degeneration,Polidocanol,disease -> disease_protein -> gene/protein -> ...,macular degeneration -> disease_protein -> SQS...,0.825303,0.993007,example
2,macular degeneration,Polidocanol,disease -> indication -> drug -> drug_effect -...,macular degeneration -> indication -> Vertepor...,1.139368,0.993007,example
